In [1]:
import ast
import math
import pandas as pd
import osmnx as ox

DATA_FILE = "osm_segment_metadata.csv"
GRAPH_FILE = "bhubaneswar_drive.graphml"

print("Loading dataset...")
df = pd.read_csv(DATA_FILE)

print("Loading OSM graph...")
G = ox.load_graphml(GRAPH_FILE)


# ---------------------------------------------------------
# Parse dataset OSM IDs
# ---------------------------------------------------------

def parse_osmid(value):
    if pd.isna(value):
        return []

    value = str(value).strip()

    if value.startswith("[") and value.endswith("]"):
        try:
            return [str(x) for x in ast.literal_eval(value)]
        except:
            return []

    return [value]


df["osmid_list"] = df["osmid"].apply(parse_osmid)


# ---------------------------------------------------------
# Build OSM ID -> graph edge lookup
# ---------------------------------------------------------

osm_edges = {}

for u, v, k, data in G.edges(keys=True, data=True):

    osmid = data.get("osmid")

    if osmid is None:
        continue

    if isinstance(osmid, list):
        ids = [str(x) for x in osmid]
    else:
        ids = [str(osmid)]

    for oid in ids:

        if oid not in osm_edges:
            osm_edges[oid] = []

        osm_edges[oid].append({
            "u": u,
            "v": v,
            "key": k,
            "length": float(data.get("length", 0)),
            "highway": data.get("highway")
        })


# ---------------------------------------------------------
# Haversine distance
# ---------------------------------------------------------

def haversine(lat1, lon1, lat2, lon2):

    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.asin(math.sqrt(a))


# ---------------------------------------------------------
# Validate each segment
# ---------------------------------------------------------

results = []

for _, row in df.iterrows():

    segment_id = row["segment_id"]

    dataset_ids = row["osmid_list"]

    dataset_length = row["length_m"]
    dataset_lat = row["lat"]
    dataset_lon = row["lon"]
    dataset_type = str(row["road_type"])

    candidate_edges = []

    for oid in dataset_ids:

        if oid in osm_edges:
            candidate_edges.extend(osm_edges[oid])

    if not candidate_edges:

        results.append({
            "segment_id": segment_id,
            "matched": False
        })

        continue

    # Find closest edge geographically
    best_edge = None
    best_distance = float("inf")

    for edge in candidate_edges:

        node = G.nodes[edge["u"]]

        edge_lat = float(node["y"])
        edge_lon = float(node["x"])

        distance = haversine(
            dataset_lat,
            dataset_lon,
            edge_lat,
            edge_lon
        )

        if distance < best_distance:
            best_distance = distance
            best_edge = edge

    osm_length = best_edge["length"]

    length_error = abs(dataset_length - osm_length)

    length_error_pct = (
        length_error / dataset_length * 100
        if dataset_length > 0
        else None
    )

    results.append({
        "segment_id": segment_id,
        "matched": True,
        "dataset_length_m": dataset_length,
        "osm_length_m": osm_length,
        "length_error_m": length_error,
        "length_error_pct": length_error_pct,
        "coordinate_distance_m": best_distance,
        "dataset_road_type": dataset_type,
        "osm_road_type": str(best_edge["highway"]),
        "matched_osmid": ",".join(dataset_ids)
    })


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

results_df = pd.DataFrame(results)

matched = results_df[
    results_df["matched"] == True
].copy()

print("\n========================================")
print("OSM CONSISTENCY VALIDATION")
print("========================================")

print("\nTotal segments:", len(results_df))
print("Matched:", len(matched))
print(
    "Match rate:",
    f"{len(matched) / len(results_df) * 100:.2f}%"
)

print("\n========== LENGTH ==========")

print(
    "Mean length error:",
    f"{matched['length_error_m'].mean():.2f} m"
)

print(
    "Median length error:",
    f"{matched['length_error_m'].median():.2f} m"
)

print(
    "Mean percentage error:",
    f"{matched['length_error_pct'].mean():.2f}%"
)

print(
    "Median percentage error:",
    f"{matched['length_error_pct'].median():.2f}%"
)

print(
    "Segments within 10%:",
    f"{(matched['length_error_pct'] <= 10).sum()} / {len(matched)}"
)

print(
    "Segments within 20%:",
    f"{(matched['length_error_pct'] <= 20).sum()} / {len(matched)}"
)


print("\n========== COORDINATES ==========")

print(
    "Mean coordinate distance:",
    f"{matched['coordinate_distance_m'].mean():.2f} m"
)

print(
    "Median coordinate distance:",
    f"{matched['coordinate_distance_m'].median():.2f} m"
)

print(
    "Within 100 m:",
    f"{(matched['coordinate_distance_m'] <= 100).sum()} / {len(matched)}"
)

print(
    "Within 500 m:",
    f"{(matched['coordinate_distance_m'] <= 500).sum()} / {len(matched)}"
)


print("\n========== ROAD TYPES ==========")

print(
    pd.crosstab(
        matched["dataset_road_type"],
        matched["osm_road_type"]
    )
)


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

results_df.to_csv(
    "osm_consistency_results.csv",
    index=False
)

print("\nSaved:")
print("osm_consistency_results.csv")

Loading dataset...
Loading OSM graph...

OSM CONSISTENCY VALIDATION

Total segments: 700
Matched: 700
Match rate: 100.00%

========== LENGTH ==========
Mean length error: 66.47 m
Median length error: 0.00 m
Mean percentage error: 37.01%
Median percentage error: 0.00%
Segments within 10%: 417 / 700
Segments within 20%: 449 / 700

========== COORDINATES ==========
Mean coordinate distance: 108.34 m
Median coordinate distance: 64.48 m
Within 100 m: 496 / 700
Within 500 m: 680 / 700

========== ROAD TYPES ==========
osm_road_type      primary  secondary  trunk
dataset_road_type                           
primary                334          0      0
secondary                0        275      0
trunk                    0          0     91

Saved:
osm_consistency_results.csv
